# Practical 06 — Credit Risk

The agentic project drives a **credit-memo agent**: it loads a company's raw
financial figures, computes credit ratios, scores default risk with a transparent
rule-based model, and drafts a memo that cites every number it states. The agent
never does arithmetic or recalls a figure from memory — the deterministic tools in
`tools/` compute everything.

Colab can't run the agentic Claude Code / Cline loop, so this notebook drives the
**reference tools directly**, end to end, fully offline on the bundled fictional
companies: **load → ratios → score → memo**. You will see the real outputs of each
tool and assemble a grounded memo exactly the way the agent is required to.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "06-credit-risk"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## Step 0 — Import the deterministic tools

Three modules do all the real work. We import their public functions directly
(the Colab counterpart of the agent shelling out to `python -m tools.*`):

- `tools._common.list_companies` / `load_financials` — discover and load fixtures.
- `tools.ratios.compute_ratios` / `ratios_for` — leverage, coverage, liquidity.
- `tools.score.score_company` / `score_for` / `risk_components` — the 0–100 risk
  score, the LOW/MEDIUM/HIGH flag, and the three monotone-in-risk sub-scores.

In [ ]:
import json
from tools._common import list_companies, load_financials
from tools.ratios import compute_ratios, ratios_for
from tools.score import score_company, score_for, risk_components

print("Available companies:", list_companies())

## Step 1 — Load a company's raw figures

`load_financials` reads `data/companies/<slug>.json`. These are the only numbers in
the whole pipeline; every ratio and score is derived from them. We'll analyse
`cobalt`, the distressed specialty retailer, as the running example.

In [ ]:
company = "cobalt"
fin = load_financials(company)
print(json.dumps(fin, indent=2))

## Step 2 — Compute credit ratios

`compute_ratios` turns the raw figures into the standard credit ratios. Note the
design choice: any ratio that would divide by zero (no interest expense, no current
liabilities, no debt) returns `null` rather than a bogus or infinite number.

- **leverage** = gross debt / EBITDA (lower is safer)
- **net_leverage** = (debt − cash) / EBITDA (lower is safer)
- **interest_coverage** = EBITDA / interest (higher is safer)
- **current_ratio** = current assets / current liabilities (higher is safer)
- **cash_to_debt** = cash / total debt (higher is safer)

In [ ]:
ratios = compute_ratios(fin)   # same as ratios_for(company)
print(json.dumps(ratios, indent=2))

## Step 3 — Score default risk

`score_company` blends three sub-scores, each in [0, 1] and each *monotone in risk*
(a weaker company always scores at least as high): leverage (weight 0.40), coverage
(0.35), liquidity (0.25). The blended value is scaled to 0–100 and bucketed into a
flag: `< 25` LOW, `< 55` MEDIUM, else HIGH. `default_risk` is `True` only when HIGH.

In [ ]:
score = score_company(fin)     # same as score_for(company)
print(json.dumps(score, indent=2))

## Step 4 — Draft the grounded credit memo

This is the agent's job: interpret — never recompute — the tool outputs. The rule is
strict: **every number stated must be one a tool just printed**, and a `null` ratio
must be called *undefined*, not guessed. Below we assemble exactly such a memo
programmatically from the `ratios` and `score` dicts, so by construction it cannot
invent a figure.

In [ ]:
def fmt(x, suffix=""):
    """Render a ratio for the memo, honouring null = undefined."""
    return "undefined" if x is None else f"{x}{suffix}"

def build_memo(fin, ratios, score):
    lines = []
    lines.append(f"# Credit Memo — {fin['name']} ({fin['company']})")
    lines.append("")
    lines.append(f"_Sector: {fin.get('sector', 'n/a')} · FY{fin.get('fiscal_year','?')} "
                 f"· figures in {fin.get('currency','units')}_")
    lines.append("")
    lines.append("## Risk assessment")
    lines.append(
        f"Default-risk score **{score['risk_score']}/100**, flag "
        f"**{score['risk_flag']}** (score). `default_risk` is "
        f"{'set' if score['default_risk'] else 'not set'}.")
    lines.append("")
    lines.append("## Key ratios (ratios)")
    lines.append(f"- Net debt: {ratios['net_debt']}")
    lines.append(f"- Gross leverage (debt/EBITDA): {fmt(ratios['leverage'], 'x')}")
    lines.append(f"- Net leverage (net debt/EBITDA): {fmt(ratios['net_leverage'], 'x')}")
    lines.append(f"- Interest coverage (EBITDA/interest): {fmt(ratios['interest_coverage'], 'x')}")
    lines.append(f"- Current ratio: {fmt(ratios['current_ratio'])}")
    lines.append(f"- Cash / total debt: {fmt(ratios['cash_to_debt'])}")
    lines.append("")
    lines.append("## Sub-scores (score, 0 = safe, 1 = max concern)")
    c = score["components"]
    w = score["weights"]
    for k in ("leverage", "coverage", "liquidity"):
        lines.append(f"- {k}: {c[k]} (weight {w[k]})")
    lines.append("")
    lines.append("## Interpretation")
    cov = ratios["interest_coverage"]
    if cov is None:
        lines.append("- Interest coverage is undefined (no interest expense): no coverage burden.")
    elif cov < 2:
        lines.append(f"- Coverage {cov}x is thin — earnings barely service interest.")
    else:
        lines.append(f"- Coverage {cov}x leaves comfortable headroom over interest.")
    lines.append(f"- The {score['risk_flag']} flag follows directly from the score band; "
                 "it is not re-weighted here.")
    return "\n".join(lines)

memo = build_memo(fin, ratios, score)
print(memo)

## Step 5 — Save the memo to `reports/`

The agent ends by writing `reports/<company>.md`. We do the same so the artifact
exists on disk, just like the reference workflow.

In [ ]:
reports_dir = root / "reports"
reports_dir.mkdir(exist_ok=True)
out_path = reports_dir / f"{company}.md"
out_path.write_text(memo + "\n", encoding="utf-8")
print("Wrote", out_path)

## Things to try (1) — Compare across the risk spectrum

The README claims the score is **monotone in risk**: a weaker balance sheet must
score higher. Run every bundled company through `score_for` and confirm the ordering
matches their profiles: aurora (strong) < borealis (mid) < cobalt (distressed). Delta
is debt-free, so it should land LOW too.

In [ ]:
print(f"{'company':10} {'score':>6}  flag    default_risk")
print("-" * 40)
for slug in list_companies():
    s = score_for(slug)
    print(f"{slug:10} {s['risk_score']:>6}  {s['risk_flag']:<7} {s['default_risk']}")

Observation: cobalt scores the highest and is the only HIGH / `default_risk = True`
firm, while aurora scores the lowest — exactly the monotonicity the scorer guarantees.

## Things to try (2) — A debt-free firm has undefined coverage

`delta` has no debt and no interest expense. Its `interest_coverage` and
`cash_to_debt` must come back `null`, and the score must treat those as *no burden*
(sub-scores 0), not invent a number — the memo must say "undefined".

In [ ]:
delta_ratios = ratios_for("delta")
delta_score = score_for("delta")
print("delta interest_coverage:", delta_ratios["interest_coverage"])
print("delta cash_to_debt     :", delta_ratios["cash_to_debt"])
print("delta sub-scores       :", json.dumps(delta_score["components"]))
print("delta flag             :", delta_score["risk_flag"])

## Things to try (3) — Edit a figure and watch the score move

The README suggests halving cobalt's `interest_expense` and re-scoring. We do it on an
in-memory copy (no files touched): less interest means stronger coverage, so the
coverage sub-score and the overall risk score should both fall.

In [ ]:
cobalt = load_financials("cobalt")
before = score_company(cobalt)

cobalt_edit = dict(cobalt)
cobalt_edit["interest_expense"] = cobalt["interest_expense"] / 2  # 95 -> 47.5

after = score_company(cobalt_edit)

print(f"interest_expense  {cobalt['interest_expense']} -> {cobalt_edit['interest_expense']}")
print(f"coverage ratio    {ratios_for('cobalt')['interest_coverage']} -> "
      f"{compute_ratios(cobalt_edit)['interest_coverage']}")
print(f"coverage sub-score {before['components']['coverage']} -> {after['components']['coverage']}")
print(f"risk score        {before['risk_score']} ({before['risk_flag']}) -> "
      f"{after['risk_score']} ({after['risk_flag']})")

## Recap / next steps

We drove the credit-risk reference tools end to end, fully offline:

1. **Load** raw figures with `load_financials` — the only numbers in the pipeline.
2. **Ratios** with `compute_ratios` — leverage, coverage, liquidity, with `null` for
   undefined (divide-by-zero) ratios.
3. **Score** with `score_company` — a transparent, weighted, monotone-in-risk 0–100
   score plus a LOW/MEDIUM/HIGH flag.
4. **Memo** — a grounded write-up that cites only tool-produced numbers and labels
   `null` ratios as undefined, saved under `reports/`.

We also confirmed the score's monotonicity across companies, that a debt-free firm
yields undefined coverage, and that editing an input moves the score the expected way.

Next steps mirroring the README: add a fifth company JSON under `data/companies/` and
rerun the cells; try stating a ratio the tools never printed and notice you can't —
every number here traces back to a tool. In the full agentic project, the
`ratio-checker` sub-agent enforces exactly that, bouncing back any memo with an
uncited figure.